# 전국 최종감염농장 통합

`최종감염농장/{시도}_최종감염농장.csv` 전체를 합치고, `LVSTCKSPC_CODE`로 `livestock_codes.csv`를 찾아 `축종명`/`상세구분`을 덮어쓴다 (기존 값이 있어도 코드 기준으로 다시 채움). 컬럼명을 영문 스네이크케이스로 바꿔서 `전국_최종감염농장.csv`로 저장한다. `livestock_codes.csv`에 없는 코드는 따로 보고한다.

## 1. `최종감염농장/*_최종감염농장.csv` 전체 통합

In [1]:
import glob
import os

import pandas as pd

FINAL_DIR = "최종감염농장"
OUTPUT_PATH = "전국_최종감염농장.csv"

# 패턴에 한글을 넣으면 macOS가 파일명을 NFD(분해형)로 저장한 파일(예: "경기도_...")이
# NFC 패턴과 바이트 단위로 안 맞아 누락될 수 있어, ASCII만으로 된 패턴("*.csv")을 사용한다.
csv_files = sorted(glob.glob(os.path.join(FINAL_DIR, "*.csv")))
print(f"통합 대상 파일 {len(csv_files)}개")
for path in csv_files:
    print(" -", os.path.basename(path))

frames = [pd.read_csv(path, encoding="utf-8-sig", dtype={"LVSTCKSPC_CODE": str}) for path in csv_files]
all_df = pd.concat(frames, ignore_index=True)
print(f"\n총 {len(all_df)}행 통합 완료")


통합 대상 파일 5개
 - 경기도_최종감염농장.csv
 - 전라남도_최종감염농장.csv
 - 전라북도_최종감염농장.csv
 - 충청남도_최종감염농장.csv
 - 충청북도_최종감염농장.csv

총 1173행 통합 완료


## 2. `LVSTCKSPC_CODE` 기준으로 `축종명`/`상세구분` 덮어쓰기

`livestock_codes.csv`(표준 축산코드 → 축종명/상세구분)를 참조해서 코드 기준으로 다시 채운다. 기존 값이 있어도 코드 기준 값으로 덮어쓰고, `livestock_codes.csv`에 없는 코드는 원본 값을 그대로 두고 따로 보고한다.

In [2]:
codes_df = pd.read_csv("livestock_codes.csv", dtype=str)
code_map = codes_df.set_index("표준 축산코드")[["축종명", "상세구분"]].to_dict("index")

codes_in_data = set(all_df["LVSTCKSPC_CODE"].dropna().astype(str))
missing_codes = sorted(codes_in_data - set(code_map))

if missing_codes:
    print(f"⚠️  livestock_codes.csv에 없는 코드 {len(missing_codes)}개 (해당 행은 원본 값 유지):")
    for c in missing_codes:
        print(" -", c)
else:
    print("✓ livestock_codes.csv에 없는 코드 없음 (모든 코드 매핑 가능)")


✓ livestock_codes.csv에 없는 코드 없음 (모든 코드 매핑 가능)


In [3]:
def lookup(code, field):
    info = code_map.get(str(code))
    return info[field] if info is not None else None

has_code = all_df["LVSTCKSPC_CODE"].astype(str).isin(code_map)

# 코드가 매핑표에 있는 행만 축종명/상세구분을 코드 기준 값으로 덮어쓴다 (없는 코드는 원본 유지)
all_df.loc[has_code, "축종명"] = all_df.loc[has_code, "LVSTCKSPC_CODE"].map(lambda c: lookup(c, "축종명"))
all_df.loc[has_code, "상세구분"] = all_df.loc[has_code, "LVSTCKSPC_CODE"].map(lambda c: lookup(c, "상세구분"))

print(f"코드 매핑으로 덮어쓴 행: {has_code.sum()} / {len(all_df)}")


코드 매핑으로 덮어쓴 행: 1173 / 1173


## 3. `FARM_LOCPLC`를 매칭된 `소재지지번주소`로 덮어쓰기

원본 파이프라인은 감염농장명(`FARM_NM`)과 농장현황명(`농장명`)이 **정확히** 같을 때만 주소를 덮어썼다. 그래서 "노O호"(농장현황의 익명화 표기) ↔ "노영호"(감염농장 원본명)처럼 유사매칭(이름 유사도 ≥60%)으로만 연결된 농장은 `소재지지번주소`는 채워졌는데 `FARM_LOCPLC`는 덜 정확한 원본 주소로 남아있었다. `소재지지번주소`가 있는 행(=매칭에 성공한 행)은 모두 `FARM_LOCPLC`를 그 값으로 덮어쓴다.

In [4]:
has_jibun = all_df["소재지지번주소"].notna()
overwritten = has_jibun & (all_df["FARM_LOCPLC"] != all_df["소재지지번주소"])
print(f"FARM_LOCPLC를 소재지지번주소로 덮어쓴 행: {overwritten.sum()} / {has_jibun.sum()}(매칭된 행)")

all_df.loc[has_jibun, "FARM_LOCPLC"] = all_df.loc[has_jibun, "소재지지번주소"]


FARM_LOCPLC를 소재지지번주소로 덮어쓴 행: 0 / 306(매칭된 행)


## 4. 컬럼명 영문화 + 저장

In [5]:
COLUMN_RENAME = {
    "시군명": "county",
    "농장명": "farm_name",
    "축종명": "livestock_name",
    "상세구분": "livestock_type",
    "사육두수(마리)": "head_count",
    "소재지지번주소": "jibun_address",
    "WGS84위도": "latitude",
    "WGS84경도": "longitude",
    "FARM_NM": "farm_alias",
    "FARM_LOCPLC": "farm_address",
    "OCCRRNC_DE": "outbreak_date",
    "LVSTCKSPC_CODE": "livestock_code",
}

final_df = all_df[list(COLUMN_RENAME)].rename(columns=COLUMN_RENAME)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"✓ {OUTPUT_PATH} 저장 완료 ({len(final_df)}행, {len(final_df.columns)}컬럼)")
final_df.head()


✓ 전국_최종감염농장.csv 저장 완료 (1173행, 12컬럼)


,county,farm_name,livestock_name,livestock_type,head_count,jibun_address,latitude,longitude,farm_alias,farm_address,outbreak_date,livestock_code
0,고양시,노O호,닭,토종닭,3000.0,경기도 고양시 덕양구 관산동 388-2,126.853119,37.703825,노영호,경기도 고양시 덕양구 관산동 388-2,20170307,415002
1,고양시,장항양계,닭,산란계,37000.0,경기도 고양시 일산서구 구산동 681번지,37.688283,126.692534,장항양계,경기도 고양시 일산서구 구산동 681번지,20230108,415003
2,광주시,NaN,오리,기타,NaN,NaN,NaN,NaN,NaN,경기도 광주시 남한산성면 불당리,20160405,416299
3,광주시,광주농장,닭,산란계,59000.0,경기도 광주시 초월읍 신월리 409-1,37.410803,127.307538,광주,경기도 광주시 초월읍 신월리 409-1,20161222,415003
4,김포시,(주)한우리,닭,산란계,NaN,NaN,NaN,NaN,(주)한우리,경기도 김포시 통진읍 가현리,20150203,415003
